# Data Preparation and Data Split

In this notebook, the data is first split into a train and test set. 

In [ ]:
#importing libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import MinMaxScaler
import pickle

# Random Forest feature importance 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.inspection import permutation_importance

### 1. Load the preprocessed Data

Load the patient_data.pkl file that not only has the values saved, but also the categories and so on.

In [ ]:
# Loading the dataset
with open('data/patient_data.pkl', 'rb') as f:
    data = pickle.load(f)

# Display data types
print(data.dtypes)

Patient                        int64
hospital                    category
age                          float64
sex                         category
rosc                         float64
ohca                        category
shockable_rhythm            category
ttm                         category
outcome                     category
cpc                         category
rosc_missing                   int64
shockable_rhythm_missing       int64
dtype: object


In [ ]:
# checking unique categories of categorical columns
cat_col = ['hospital', 'sex', 'ohca', 'ttm', 'shockable_rhythm', 'cpc']

for col in cat_col:
    unique_values = data[col].unique()
    print(f"Unique values in '{col}': {unique_values}")

Unique values in 'hospital': ['A', 'F', 'D', 'E', 'B']
Categories (5, object): ['A', 'B', 'D', 'E', 'F']
Unique values in 'sex': ['Male', 'Female', NaN]
Categories (2, object): ['Female', 'Male']
Unique values in 'ohca': [True, False, 'Unknown']
Categories (3, object): [False, True, 'Unknown']
Unique values in 'ttm': [33.0, 'No TTM', 36.0]
Categories (3, object): [33.0, 36.0, 'No TTM']
Unique values in 'shockable_rhythm': [True, False, 'Unknown']
Categories (3, object): [False, True, 'Unknown']
Unique values in 'cpc': [1, 2, 5, 3, 4]
Categories (5, int64): [1 < 2 < 3 < 4 < 5]


### 2. Make Data Machine-Readable

This step involves a few sub-steps. First, standardize or normalize the numeric data to ensure consistency in scaling. Additionally, apply methods like one-hot encoding to the categorical features for effective machine processing.


#### 2.1 Impute Missing ROSC Times with an Informed Approach

ROSC time is significantly influenced by whether the cardiac arrest occurred in-hospital or out-of-hospital, as response times vary greatly in these settings. Additionally, the outcome of a cardiac arrest event may also be related to the ROSC time. Therefore, we use both the OHCA (Out-of-Hospital Cardiac Arrest) status and the outcome to calculate the median ROSC time for each category, which is then used as the imputation value for missing entries. This approach ensures that the imputed values are more representative of the underlying conditions.


In [ ]:
# Filter valid data where ROSC is not missing
valid_data = data[data['rosc'].notna()]

# Calculate median ROSC time for each OHCA and Outcome category
no_ohca_good_median = valid_data[(valid_data['ohca'] == False) & (valid_data['outcome'] == 'Good')]['rosc'].median()
no_ohca_bad_median = valid_data[(valid_data['ohca'] == False) & (valid_data['outcome'] == 'Poor')]['rosc'].median()
ohca_good_median = valid_data[(valid_data['ohca'] == True) & (valid_data['outcome'] == 'Good')]['rosc'].median()
ohca_bad_median = valid_data[(valid_data['ohca'] == True) & (valid_data['outcome'] == 'Poor')]['rosc'].median()
unknown_ohca_good_median = valid_data[(valid_data['ohca'] == 'Unknown') & (valid_data['outcome'] == 'Good')]['rosc'].median()
unknown_ohca_poor_median = valid_data[(valid_data['ohca'] == 'Unknown') & (valid_data['outcome'] == 'Poor')]['rosc'].median()

# Print median values for verification
print(f"No OHCA - Good Outcome Median: {no_ohca_good_median}")
print(f"No OHCA - Bad Outcome Median: {no_ohca_bad_median}")
print(f"OHCA - Good Outcome Median: {ohca_good_median}")
print(f"OHCA - Bad Outcome Median: {ohca_bad_median}")
print(f"Unknown OHCA - Good Outcome Median: {unknown_ohca_good_median}")
print(f"Unknown OHCA - Poor Outcome Median: {unknown_ohca_poor_median}")

No OHCA - Good Outcome Median: 11.0
No OHCA - Bad Outcome Median: 15.0
OHCA - Good Outcome Median: 18.0
OHCA - Bad Outcome Median: 21.5
Unknown OHCA - Good Outcome Median: 15.0
Unknown OHCA - Poor Outcome Median: 15.0


In [ ]:
def impute_rosc(row):
    if pd.isna(row['rosc']):
        if row['ohca'] == False and row['outcome'] == 'Good':
            return no_ohca_good_median
        elif row['ohca'] == False and row['outcome'] == 'Poor':
            return no_ohca_bad_median
        elif row['ohca'] == True and row['outcome'] == 'Good':
            return ohca_good_median
        elif row['ohca'] == True and row['outcome'] == 'Poor':
            return ohca_bad_median
        elif row['ohca'] == 'Unknown' and row['outcome'] == 'Good':
            return unknown_ohca_good_median
        elif row['ohca'] == 'Unknown' and row['outcome'] == 'Poor':
            return unknown_ohca_poor_median
    else:
        return row['rosc']

# Apply the updated imputation function to the dataset
data['rosc'] = data.apply(impute_rosc, axis=1)

# Check the number of missing values after imputation
print("Number of missing ROSC values after imputation:", data['rosc'].isna().sum())

Number of missing ROSC values after imputation: 0


#### 2.2 One-Hot Encoding for Categorical Data

We begin by examining the categorical features to determine if they are ordinal (i.e., have a meaningful order) or nominal (i.e., no inherent order). If the categories are not ordinal, we apply one-hot encoding to effectively transform them into a machine-readable format.


In [ ]:
# Define the categorical columns excluding the 'cpc' column
cat_col_excluding_cpc = [col for col in cat_col if col != 'cpc']

# Apply one-hot encoding only to non-ordinal categorical columns
oh_cat_col = pd.get_dummies(data[cat_col_excluding_cpc])
oh_cat_col

,hospital_A,hospital_B,hospital_D,hospital_E,hospital_F,sex_Female,sex_Male,ohca_False,ohca_True,ohca_Unknown,ttm_33.0,ttm_36.0,ttm_No TTM,shockable_rhythm_False,shockable_rhythm_True,shockable_rhythm_Unknown
0,True,False,False,False,False,False,True,False,True,False,True,False,False,False,True,False
1,False,False,False,False,True,True,False,True,False,False,False,False,True,True,False,False
2,True,False,False,False,False,False,True,False,True,False,False,True,False,False,True,False
3,True,False,False,False,False,False,True,False,True,False,True,False,False,False,True,False
4,False,False,True,False,False,False,True,False,True,False,True,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602,False,False,False,True,False,False,True,False,True,False,True,False,False,True,False,False
603,False,False,False,False,True,False,True,False,True,False,False,False,True,True,False,False
604,False,False,False,True,False,False,True,False,True,False,False,True,False,False,True,False
605,True,False,False,False,False,False,True,False,True,False,False,True,False,False,True,False


In [ ]:
# Concatenating the one-hot encoded dataframe with the numerical one 
numerical_col = ['Patient', 'age', 'rosc']
data_encoded = pd.concat([data[numerical_col], oh_cat_col], axis=1)

data_encoded



,Patient,age,rosc,hospital_A,hospital_B,hospital_D,hospital_E,hospital_F,sex_Female,sex_Male,ohca_False,ohca_True,ohca_Unknown,ttm_33.0,ttm_36.0,ttm_No TTM,shockable_rhythm_False,shockable_rhythm_True,shockable_rhythm_Unknown
0,284,53.0,18.0,True,False,False,False,False,False,True,False,True,False,True,False,False,False,True,False
1,286,85.0,7.0,False,False,False,False,True,True,False,True,False,False,False,False,True,True,False,False
2,296,48.0,18.0,True,False,False,False,False,False,True,False,True,False,False,True,False,False,True,False
3,299,45.0,18.0,True,False,False,False,False,False,True,False,True,False,True,False,False,False,True,False
4,303,51.0,24.0,False,False,True,False,False,False,True,False,True,False,True,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
602,1016,87.0,7.0,False,False,False,True,False,False,True,False,True,False,True,False,False,True,False,False
603,1017,26.0,52.0,False,False,False,False,True,False,True,False,True,False,False,False,True,True,False,False
604,1018,63.0,21.5,False,False,False,True,False,False,True,False,True,False,False,True,False,False,True,False
605,1019,72.0,18.0,True,False,False,False,False,False,True,False,True,False,False,True,False,False,True,False


In [ ]:
# Save the DataFrame to a Pickle file
with open('data/patient_data_onehot_imputed.pkl', 'wb') as file:
    pickle.dump(data_encoded, file)

# Confirmation message
print("DataFrame has been saved to 'data/patient_data_onehot_imputed.pkl'")

DataFrame has been saved to 'data/patient_data_onehot_imputed.pkl'
